# Motion-S Dataset Analysis

This notebook provides comprehensive analysis of the sign language motion dataset including:
- Dataset structure and statistics
- Text and gloss analysis
- Motion feature analysis
- Quality checks and anomaly detection

## 1. Setup and Configuration

In [ ]:
%%capture 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings

# --- Environment and Visuals ---
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# --- Updated Path Configuration ---
IS_KAGGLE = Path("/kaggle/working").exists()

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language")
    MODEL_DIR = Path("/kaggle/input/motion-s-vae-rvq/pytorch/default/3")
    
    DATASET_ROOT = INPUT_DIR / "Train"
    # If you are READING pre-processed features:
    MOTION_FEATS_DIR = INPUT_DIR / "Motion-Features" 
    # If you are GENERATING new features, use this instead:
    # MOTION_FEATS_DIR = Path("/kaggle/working/Motion-Features")
    
    TRAIN_CSV = INPUT_DIR / "train.csv"
    MODEL_PATH = MODEL_DIR / "rvq_vae_best.pth"
    NORM_PATH = INPUT_DIR / "normalization.npz"
    print("Running on Kaggle")
else:
    PROJECT_ROOT = Path("..").resolve()
    DATASET_ROOT = PROJECT_ROOT / "Train"
    MOTION_FEATS_DIR = PROJECT_ROOT / "Motion-Features"
    TRAIN_CSV = PROJECT_ROOT / "train.csv"
    MODEL_PATH = PROJECT_ROOT / "models" / "rvq_vae_best.pth"
    NORM_PATH = PROJECT_ROOT / "normalization.npz"
    print(f"Running locally at {PROJECT_ROOT}")

# --- Helper Function ---
def get_bvh_path(inp):
    """Handles both raw IDs and the //dataset/ paths from the CSV."""
    inp_str = str(inp)
    if "dataset" in inp_str:
        clean_path = inp_str.replace("//dataset/", "").replace("dataset/", "").lstrip("/")
        return DATASET_ROOT / clean_path
    return DATASET_ROOT / inp_str / f"{inp_str}.bvh"




In [ ]:
# Helper functions

def load_metadata(sample_dir: Path) -> dict:
    """Load metadata.txt from a sample directory."""
    metadata_path = sample_dir / "metadata.txt"
    if not metadata_path.exists():
        return None
    
    result = {}
    try:
        with open(metadata_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith("SENTENCE:"):
                    result['sentence'] = line.replace("SENTENCE:", "").strip()
                elif line.startswith("GLOSS:"):
                    result['gloss'] = line.replace("GLOSS:", "").strip()
    except Exception as e:
        return None
    
    return result if 'sentence' in result and 'gloss' in result else None


def count_bvh_files(sample_dir: Path) -> int:
    """Count BVH files in a sample directory."""
    return len(list(sample_dir.glob("*.bvh")))


def parse_glosses(gloss_str: str) -> list:
    """Parse gloss string into list of individual glosses."""
    # Remove sentence boundary markers and split
    cleaned = gloss_str.replace("//", "").strip()
    return [g.strip() for g in cleaned.split() if g.strip()]


def is_fingerspelling(gloss: str) -> bool:
    """Check if a gloss is a fingerspelled letter (single uppercase letter)."""
    return len(gloss) == 1 and gloss.isupper()

## 2. Dataset Overview

In [ ]:
# Load all sample data into a dataframe
samples_data = []

if DATASET_ROOT.exists():
    sample_dirs = [d for d in DATASET_ROOT.iterdir() if d.is_dir()]
    
    for sample_dir in tqdm(sample_dirs, desc="Loading samples"):
        sample_id = sample_dir.name
        metadata = load_metadata(sample_dir)
        bvh_count = count_bvh_files(sample_dir)
        
        if metadata:
            glosses = parse_glosses(metadata['gloss'])
            fingerspell_count = sum(1 for g in glosses if is_fingerspelling(g))
            
            samples_data.append({
                'sample_id': sample_id,
                'sentence': metadata['sentence'],
                'gloss': metadata['gloss'],
                'gloss_list': glosses,
                'gloss_count': len(glosses),
                'fingerspell_count': fingerspell_count,
                'bvh_count': bvh_count,
                'sentence_word_count': len(metadata['sentence'].split()),
                'sentence_char_count': len(metadata['sentence']),
            })
        else:
            samples_data.append({
                'sample_id': sample_id,
                'sentence': None,
                'gloss': None,
                'gloss_list': [],
                'gloss_count': 0,
                'fingerspell_count': 0,
                'bvh_count': bvh_count,
                'sentence_word_count': 0,
                'sentence_char_count': 0,
            })

df = pd.DataFrame(samples_data)
print(f"Loaded {len(df)} samples")

In [ ]:
# Basic statistics
valid_df = df[df['sentence'].notna()]

print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)
print(f"Total samples:           {len(df):,}")
print(f"Valid samples (w/ meta): {len(valid_df):,}")
print(f"Missing metadata:        {len(df) - len(valid_df):,}")
print(f"Total BVH files:         {df['bvh_count'].sum():,}")
print()
print("BVH files per sample:")
print(f"  Mean:   {df['bvh_count'].mean():.1f}")
print(f"  Median: {df['bvh_count'].median():.1f}")
print(f"  Min:    {df['bvh_count'].min()}")
print(f"  Max:    {df['bvh_count'].max()}")

## 3. Text and Linguistic Analysis

### 3.1 Sentence Statistics

In [ ]:
# Sentence length statistics
print("SENTENCE STATISTICS")
print("-" * 40)
print(f"Word count:")
print(f"  Mean:   {valid_df['sentence_word_count'].mean():.1f}")
print(f"  Median: {valid_df['sentence_word_count'].median():.1f}")
print(f"  Min:    {valid_df['sentence_word_count'].min()}")
print(f"  Max:    {valid_df['sentence_word_count'].max()}")
print()
print(f"Character count:")
print(f"  Mean:   {valid_df['sentence_char_count'].mean():.1f}")
print(f"  Median: {valid_df['sentence_char_count'].median():.1f}")
print(f"  Min:    {valid_df['sentence_char_count'].min()}")
print(f"  Max:    {valid_df['sentence_char_count'].max()}")

# Build vocabulary
all_words = []
for sentence in valid_df['sentence']:
    words = sentence.lower().split()
    # Remove punctuation
    words = [''.join(c for c in w if c.isalnum()) for w in words]
    words = [w for w in words if w]
    all_words.extend(words)

word_counts = Counter(all_words)
print(f"\nVocabulary size: {len(word_counts):,} unique words")

In [ ]:
# Sentence length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(valid_df['sentence_word_count'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sentence Length Distribution (Words)')
axes[0].axvline(valid_df['sentence_word_count'].mean(), color='red', linestyle='--', label=f'Mean: {valid_df["sentence_word_count"].mean():.1f}')
axes[0].legend()

axes[1].hist(valid_df['sentence_char_count'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sentence Length Distribution (Characters)')
axes[1].axvline(valid_df['sentence_char_count'].mean(), color='red', linestyle='--', label=f'Mean: {valid_df["sentence_char_count"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top 30 most common words
top_words = word_counts.most_common(30)
words, counts = zip(*top_words)

plt.figure(figsize=(14, 6))
plt.bar(words, counts, edgecolor='black', alpha=0.7)
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.title('Top 30 Most Common Words in Sentences')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nTop 30 words:")
for i, (word, count) in enumerate(top_words, 1):
    print(f"  {i:2}. {word:15} {count:,}")

### 3.2 Gloss Statistics

In [ ]:
# Gloss statistics
print("GLOSS STATISTICS")
print("-" * 40)
print(f"Gloss count per sample:")
print(f"  Mean:   {valid_df['gloss_count'].mean():.1f}")
print(f"  Median: {valid_df['gloss_count'].median():.1f}")
print(f"  Min:    {valid_df['gloss_count'].min()}")
print(f"  Max:    {valid_df['gloss_count'].max()}")

# Build gloss vocabulary
all_glosses = []
for gloss_list in valid_df['gloss_list']:
    all_glosses.extend(gloss_list)

gloss_counts = Counter(all_glosses)
print(f"\nUnique glosses: {len(gloss_counts):,}")
print(f"Total gloss occurrences: {len(all_glosses):,}")

# Fingerspelling analysis
fingerspell_glosses = [g for g in all_glosses if is_fingerspelling(g)]
print(f"\nFingerspelling:")
print(f"  Total fingerspelled letters: {len(fingerspell_glosses):,}")
print(f"  Percentage of all glosses: {100*len(fingerspell_glosses)/len(all_glosses):.1f}%")
print(f"  Samples with fingerspelling: {(valid_df['fingerspell_count'] > 0).sum():,} ({100*(valid_df['fingerspell_count'] > 0).mean():.1f}%)")

In [ ]:
# Gloss distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(valid_df['gloss_count'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Gloss Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Gloss Count Distribution per Sample')
axes[0].axvline(valid_df['gloss_count'].mean(), color='red', linestyle='--', label=f'Mean: {valid_df["gloss_count"].mean():.1f}')
axes[0].legend()

# Top glosses (excluding single letters)
non_fingerspell = {g: c for g, c in gloss_counts.items() if not is_fingerspelling(g)}
top_glosses = sorted(non_fingerspell.items(), key=lambda x: x[1], reverse=True)[:20]
glosses, counts = zip(*top_glosses)

axes[1].barh(list(reversed(glosses)), list(reversed(counts)), edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Gloss')
axes[1].set_title('Top 20 Most Common Glosses (excluding fingerspelling)')

plt.tight_layout()
plt.show()

### 3.3 Sentence-Gloss Relationship

In [ ]:
# Sentence-gloss compression analysis
valid_df['compression_ratio'] = valid_df['gloss_count'] / valid_df['sentence_word_count']

print("SENTENCE-GLOSS RELATIONSHIP")
print("-" * 40)
print(f"Compression ratio (gloss_count / word_count):")
print(f"  Mean:   {valid_df['compression_ratio'].mean():.2f}")
print(f"  Median: {valid_df['compression_ratio'].median():.2f}")
print(f"  Min:    {valid_df['compression_ratio'].min():.2f}")
print(f"  Max:    {valid_df['compression_ratio'].max():.2f}")

# Correlation
corr = valid_df['sentence_word_count'].corr(valid_df['gloss_count'])
print(f"\nCorrelation (word_count vs gloss_count): {corr:.3f}")

In [ ]:
# Scatter plot: sentence length vs gloss count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(valid_df['sentence_word_count'], valid_df['gloss_count'], alpha=0.3, s=10)
axes[0].set_xlabel('Sentence Word Count')
axes[0].set_ylabel('Gloss Count')
axes[0].set_title(f'Sentence Length vs Gloss Count (r={corr:.3f})')

# Add trend line
z = np.polyfit(valid_df['sentence_word_count'], valid_df['gloss_count'], 1)
p = np.poly1d(z)
x_line = np.linspace(valid_df['sentence_word_count'].min(), valid_df['sentence_word_count'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
axes[0].legend()

# Compression ratio distribution
axes[1].hist(valid_df['compression_ratio'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_xlabel('Compression Ratio (gloss/word)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Compression Ratio Distribution')
axes[1].axvline(1.0, color='red', linestyle='--', label='Ratio = 1.0')
axes[1].axvline(valid_df['compression_ratio'].mean(), color='green', linestyle='--', label=f'Mean: {valid_df["compression_ratio"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Motion Data Analysis

### 4.1 Motion Feature Files

In [ ]:
# Load motion feature statistics
motion_stats = []

if MOTION_FEATS_DIR.exists():
    npy_files = list(MOTION_FEATS_DIR.glob("*.npy"))
    print(f"Found {len(npy_files)} motion feature files")
    
    for npy_file in tqdm(npy_files, desc="Analyzing motion files"):
        try:
            motion = np.load(npy_file, mmap_mode='r')
            motion_stats.append({
                'sample_id': npy_file.stem,
                'num_frames': len(motion),
                'feature_dim': motion.shape[1] if len(motion.shape) > 1 else 1,
            })
        except Exception as e:
            print(f"Error loading {npy_file}: {e}")
            continue
    
    motion_df = pd.DataFrame(motion_stats)
    print(f"Loaded stats for {len(motion_df)} motion files")
else:
    motion_df = pd.DataFrame()
    print("Motion features directory not found")

In [ ]:
# Motion statistics
if len(motion_df) > 0:
    FRAME_RATE = 24  # fps
    motion_df['duration_sec'] = motion_df['num_frames'] / FRAME_RATE
    
    total_frames = motion_df['num_frames'].sum()
    total_duration_sec = motion_df['duration_sec'].sum()
    
    print("MOTION STATISTICS")
    print("-" * 40)
    print(f"Total motion files: {len(motion_df):,}")
    print(f"Feature dimension:  {motion_df['feature_dim'].iloc[0]}")
    print("-" * 40)
    
    print(f"TOTAL DATASET SIZE:")
    print(f"  Total Frames:     {total_frames:,}")
    print(f"  Total Duration:   {total_duration_sec/3600:.2f} hours " 
          f"({total_duration_sec/60:.1f} minutes)")
    print("-" * 40)
    
    print(f"FRAME COUNT PER FILE:")
    print(f"  Mean:    {motion_df['num_frames'].mean():.1f}")
    print(f"  Median:  {motion_df['num_frames'].median():.1f}")
    print(f"  Min:     {motion_df['num_frames'].min()}")
    print(f"  Max:     {motion_df['num_frames'].max()}")
    print()
    
    print(f"DURATION PER FILE (seconds):")
    print(f"  Mean:    {motion_df['duration_sec'].mean():.2f}s")
    print(f"  Median:  {motion_df['duration_sec'].median():.2f}s")
    print(f"  Min:     {motion_df['duration_sec'].min():.2f}s")
    print(f"  Max:     {motion_df['duration_sec'].max():.2f}s")

In [ ]:
# Motion duration distribution
if len(motion_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(motion_df['num_frames'], bins=50, edgecolor='black', alpha=0.7, color='coral')
    axes[0].set_xlabel('Number of Frames')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Motion Length Distribution (Frames)')
    axes[0].axvline(motion_df['num_frames'].mean(), color='red', linestyle='--', label=f'Mean: {motion_df["num_frames"].mean():.0f}')
    axes[0].legend()
    
    axes[1].hist(motion_df['duration_sec'], bins=50, edgecolor='black', alpha=0.7, color='coral')
    axes[1].set_xlabel('Duration (seconds)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Motion Duration Distribution')
    axes[1].axvline(motion_df['duration_sec'].mean(), color='red', linestyle='--', label=f'Mean: {motion_df["duration_sec"].mean():.1f}s')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

### 4.2 Normalization Statistics

In [ ]:
import numpy as np

# 1. Skip if we already have the result
if NORM_PATH.exists():
    norm_data = np.load(NORM_PATH)
    mean, std = norm_data['mean'], norm_data['std']
    print(f"✅ Normalization stats loaded from: {NORM_PATH}")
else:
    print("⚠️ Normalization file not found. Calculating from data (this requires reading the files)...")
    
    sum_feats = None
    sum_sq_feats = None
    total_frames = 0

    # 2. Iterate through the files you already found
    for npy_file in tqdm(npy_files, desc="Computing Mean/Std"):
        # WE DO NOT USE mmap_mode HERE. We need the real values.
        data = np.load(npy_file).astype(np.float64) 
        
        if sum_feats is None:
            sum_feats = np.zeros(data.shape[1])
            sum_sq_feats = np.zeros(data.shape[1])
            
        sum_feats += data.sum(axis=0)
        sum_sq_feats += (data**2).sum(axis=0)
        total_frames += data.shape[0]

    # 3. Final Math
    mean = sum_feats / total_frames
    variance = (sum_sq_feats / total_frames) - (mean**2)
    std = np.sqrt(np.maximum(variance, 1e-8))
    
    # Precision guard for static joints
    std[std < 1e-6] = 1.0

    # 4. Save so you never have to do this 2-minute wait again
    SAVE_PATH = Path("/kaggle/working/normalization.npz")
    np.savez(SAVE_PATH, mean=mean, std=std)
    print(f"Normalization stats saved to {SAVE_PATH}")

In [ ]:
# Visualize normalization statistics

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean distribution
axes[0, 0].bar(range(len(mean)), mean, width=1.0, alpha=0.7)
axes[0, 0].set_xlabel('Feature Index')
axes[0, 0].set_ylabel('Mean')
axes[0, 0].set_title('Feature Mean Values')

# Std distribution
axes[0, 1].bar(range(len(std)), std, width=1.0, alpha=0.7, color='orange')
axes[0, 1].set_xlabel('Feature Index')
axes[0, 1].set_ylabel('Std')
axes[0, 1].set_title('Feature Standard Deviation')

# Mean histogram
axes[1, 0].hist(mean, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Mean Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Feature Means')

# Std histogram
axes[1, 1].hist(std, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].set_xlabel('Std Value')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Feature Stds')

plt.tight_layout()
plt.show()

### 4.3 Text-Motion Relationship

In [ ]:
# Merge motion data with text data
if len(motion_df) > 0:
    merged_df = valid_df.merge(motion_df, on='sample_id', how='inner')
    print(f"Matched {len(merged_df)} samples with both text and motion data")
    
    # Correlations
    corr_word_frames = merged_df['sentence_word_count'].corr(merged_df['num_frames'])
    corr_gloss_frames = merged_df['gloss_count'].corr(merged_df['num_frames'])
    
    print(f"\nCorrelations:")
    print(f"  Word count vs Frames: {corr_word_frames:.3f}")
    print(f"  Gloss count vs Frames: {corr_gloss_frames:.3f}")
else:
    merged_df = pd.DataFrame()
    print("No motion data available for merging")

In [ ]:
# Text-motion relationship plots
if len(merged_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Gloss count vs frames (should be stronger correlation)
    axes[0].scatter(merged_df['gloss_count'], merged_df['num_frames'], alpha=0.3, s=10)
    axes[0].set_xlabel('Gloss Count')
    axes[0].set_ylabel('Number of Frames')
    axes[0].set_title(f'Gloss Count vs Motion Length (r={corr_gloss_frames:.3f})')
    
    # Add trend line
    z = np.polyfit(merged_df['gloss_count'], merged_df['num_frames'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged_df['gloss_count'].min(), merged_df['gloss_count'].max(), 100)
    axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
    axes[0].legend()
    
    # Word count vs frames
    axes[1].scatter(merged_df['sentence_word_count'], merged_df['num_frames'], alpha=0.3, s=10, color='orange')
    axes[1].set_xlabel('Sentence Word Count')
    axes[1].set_ylabel('Number of Frames')
    axes[1].set_title(f'Word Count vs Motion Length (r={corr_word_frames:.3f})')
    
    z = np.polyfit(merged_df['sentence_word_count'], merged_df['num_frames'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(merged_df['sentence_word_count'].min(), merged_df['sentence_word_count'].max(), 100)
    axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Motion anomalies
if len(motion_df) > 0:
    print("\nMOTION ANOMALIES")
    print("-" * 40)
    
    # Very short motions
    short_motions = motion_df[motion_df['num_frames'] < 24]
    print(f"6. VERY SHORT MOTIONS (<24 frames)")
    print(f"   Count: {len(short_motions)}")
    
    # Very long motions
    q99 = motion_df['num_frames'].quantile(0.99)
    long_motions = motion_df[motion_df['num_frames'] > q99]
    print(f"\n7. VERY LONG MOTIONS (>99th percentile: {q99:.0f} frames)")
    print(f"   Count: {len(long_motions)}")
    
    # Outlier detection using IQR
    Q1 = motion_df['num_frames'].quantile(0.25)
    Q3 = motion_df['num_frames'].quantile(0.75)
    IQR = Q3 - Q1
    outliers = motion_df[(motion_df['num_frames'] < Q1 - 1.5*IQR) | (motion_df['num_frames'] > Q3 + 1.5*IQR)]
    print(f"\n8. FRAME COUNT OUTLIERS (IQR method)")
    print(f"   Q1: {Q1:.0f}, Q3: {Q3:.0f}, IQR: {IQR:.0f}")
    print(f"   Outlier count: {len(outliers)}")

## 6. Sample Examples

In [ ]:
# Display sample examples
print("SAMPLE EXAMPLES")
print("=" * 50)

# Random samples
sample_examples = valid_df.sample(5, random_state=42)

for idx, row in sample_examples.iterrows():
    print(f"\nSample ID: {row['sample_id']}")
    print(f"  Sentence: {row['sentence']}")
    print(f"  Gloss:    {row['gloss']}")
    print(f"  Words: {row['sentence_word_count']}, Glosses: {row['gloss_count']}, BVH files: {row['bvh_count']}")
    print(f"  Fingerspelling letters: {row['fingerspell_count']}")

In [ ]:
# Analyze a sample motion file in detail
if MOTION_FEATS_DIR.exists():
    sample_npy = list(MOTION_FEATS_DIR.glob("*.npy"))[:1]
    if sample_npy:
        motion = np.load(sample_npy[0])
        
        print("\nSAMPLE MOTION FILE ANALYSIS")
        print("-" * 40)
        print(f"File: {sample_npy[0].name}")
        print(f"Shape: {motion.shape}")
        print(f"Dtype: {motion.dtype}")
        print()
        print(f"Value range: [{motion.min():.4f}, {motion.max():.4f}]")
        print(f"Mean: {motion.mean():.4f}")
        print(f"Std:  {motion.std():.4f}")
        
        # Per-feature statistics
        print(f"\nPer-feature statistics (first 10 features):")
        for i in range(min(10, motion.shape[1])):
            print(f"  Feature {i:3d}: mean={motion[:, i].mean():8.4f}, std={motion[:, i].std():8.4f}")

# Motion Visualization

In [1]:
!pip uninstall kiseki -y 
!pip install git+https://github.com/1997MarsRover/kiseki.git

  Cloning https://github.com/1997MarsRover/kiseki.git to /tmp/pip-req-build-66zuuk4a
  Running command git clone --filter=blob:none --quiet https://github.com/1997MarsRover/kiseki.git /tmp/pip-req-build-66zuuk4a
  Resolved https://github.com/1997MarsRover/kiseki.git to commit 07fd2bcfadb8477e985d33d4945bf86eedcff440
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kiseki: filename=kiseki-0.1.0-py3-none-any.whl size=22844 sha256=fdea5660cd48d33e993fa1c949a3f20b2f9d0a06f8b99ca67aaa89fbc8f511ef
  Stored in directory: /tmp/pip-ephem-wheel-cache-wdp9rnxq/wheels/fc/02/33/70eef3ff9a4adde0a349f0e34e67b6ed7d4ce44874125573de
Successfully built kiseki


In [2]:
from kiseki import visualize


# Focus on hands with front view
visualize("/kaggle/input/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/1000648.npy",
          focus_joints='both_hands',
          fixed_view='front',
          fps=60, display=True)

Using BVH: /usr/local/lib/python3.12/dist-packages/kiseki/sample.bvh
Loaded: 324 frames, 668 features
Reconstructing skeleton...
Reconstructed: 324 frames, 55 joints
Creating animation...
Focusing on 32 joints, radius=67.0
Fixed view: elev=0, azim=90


# Compare Motions

In [3]:
# Compare Motions
from kiseki import compare


m_a = "/kaggle/input/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/1005354.npy"
m_b = "/kaggle/input/competitions/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/1000648.npy"
# Overlay -- both skeletons on the same axes
compare(m_a,
        m_b", 
        mode="overlay", display=True)

Using BVH: /usr/local/lib/python3.12/dist-packages/kiseki/sample.bvh
Motion A: 492 frames | Motion B: 324 frames
Creating comparison animation...
Comparison mode: overlay
Frames: 324 (trimmed to shorter motion)


## 7. Summary Statistics

In [ ]:
# Summary table
summary_data = {
    'Metric': [
        'Total Samples',
        'Valid Samples (with metadata)',
        'Total BVH Files',
        'Unique Words (vocabulary)',
        'Unique Glosses',
        'Avg Sentence Length (words)',
        'Avg Gloss Count',
        'Fingerspelling Rate',
    ],
    'Value': [
        f"{len(df):,}",
        f"{len(valid_df):,}",
        f"{df['bvh_count'].sum():,}",
        f"{len(word_counts):,}",
        f"{len(gloss_counts):,}",
        f"{valid_df['sentence_word_count'].mean():.1f}",
        f"{valid_df['gloss_count'].mean():.1f}",
        f"{100*len(fingerspell_glosses)/len(all_glosses):.1f}%",
    ]
}

if len(motion_df) > 0:
    summary_data['Metric'].extend([
        'Motion Files',
        'Feature Dimension',
        'Avg Motion Length (frames)',
        'Avg Duration (seconds)',
    ])
    summary_data['Value'].extend([
        f"{len(motion_df):,}",
        f"{motion_df['feature_dim'].iloc[0]}",
        f"{motion_df['num_frames'].mean():.0f}",
        f"{motion_df['duration_sec'].mean():.2f}s",
    ])

summary_df = pd.DataFrame(summary_data)
print("DATASET SUMMARY")
print("=" * 50)
print(summary_df.to_string(index=False))

In [ ]:
# Correlation heatmap for merged data
if len(merged_df) > 0:
    corr_cols = ['sentence_word_count', 'sentence_char_count', 'gloss_count', 
                 'fingerspell_count', 'bvh_count', 'num_frames']
    corr_matrix = merged_df[corr_cols].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                fmt='.2f', square=True, linewidths=0.5)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Export analyzed data for future use (optional)
export_cols = ['sample_id', 'sentence', 'gloss', 'gloss_count', 'sentence_word_count', 
               'sentence_char_count', 'fingerspell_count', 'bvh_count']

if len(merged_df) > 0:
    export_df = merged_df[export_cols + ['num_frames', 'duration_sec']]
else:
    export_df = valid_df[export_cols]

# Uncomment to export
export_df.to_csv('dataset_analysis.csv', index=False)
# print(f"Exported {len(export_df)} samples to dataset_analysis.csv")

print(f"DataFrame ready for export: {len(export_df)} samples, {len(export_df.columns)} columns")
export_df.head()